<a href="https://colab.research.google.com/github/rohith154-vsr/ML-2-LAB/blob/main/LP1_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# generate synthetic employee data for exploratory analysis
np.random.seed(42)
n = 500
df = pd.DataFrame({
    'Department': np.random.choice(['IT', 'HR', 'Sales'], size=n),
    'Age': np.random.randint(22, 55, size=n),
    'Salary': np.random.normal(50000, 10000, size=n)
})

# inject random nulls and one extreme salary outlier to demonstrate cleaning
df.loc[np.random.choice(n, 30), 'Salary'] = np.nan
df.loc[np.random.choice(n, 20), 'Age'] = np.nan
df.loc[0, 'Salary'] = 300000

display(df.head())
print("Missing values before cleaning:
", df.isnull().sum())

# handle missing values:
# - salary median is grouped by department since compensation varies by role
# - age filled with overall median (median is robust against outliers)
df['Salary'] = df['Salary'].fillna(df.groupby('Department')['Salary'].transform('median'))
df['Age'] = df['Age'].fillna(df['Age'].median())

# clip outliers using IQR rule (values outside Q1 - 1.5*IQR to Q3 + 1.5*IQR are capped)
for col in ['Salary', 'Age']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = np.clip(df[col], Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print("
Missing values after cleaning:
", df.isnull().sum())
print("
Summary statistics after outlier clipping:")
display(df.describe())

# visualizations: distribution histogram and correlation matrix
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(df['Salary'], kde=True, color='purple')
plt.title('Salary Distribution (Clipped)')

plt.subplot(1, 2, 2)
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='Blues', fmt='.2f')
plt.title('Feature Correlation Heatmap')

plt.tight_layout()
plt.show()